# 01 - Drugs@FDA Full Audit and Master Dataset Build

## Purpose
This notebook documents a **full, reproducible audit** of the `dafdata20260313` Drugs@FDA extract and then builds an **unfiltered master dataset** for downstream analysis.

The goal here is not to jump straight to a thesis-ready subset. The goal is to answer, carefully and transparently:

1. What is in each raw FDA text file?
2. What is the unit of observation in each file?
3. Which joins are safe, and which would create duplicate inflation?
4. What can the extract tell us about applications, submission events, approvals, tentative approvals, products, and supporting metadata?
5. How do we create one workable CSV that preserves as much of the available information as possible without silently changing the data-generating structure?

## Final output
By the end of the notebook, we export:

- `data/processed/fda_backbone.csv`

In this notebook, `fda_backbone.csv` means a **master submission-event panel** with one row per `ApplNo + SubmissionType + SubmissionNo`, enriched with application-level, product-level, document, property, action-type, marketing-status, and therapeutic-equivalence information.


In [1]:
from pathlib import Path
from IPython.display import display
import csv
import pandas as pd
import numpy as np


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "dafdata20260313").exists() and (candidate / "code" / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root from the current working directory. "
        "Expected to find data/raw/dafdata20260313 and code/notebooks in the repo."
    )


ROOT = find_repo_root()
DATA = ROOT / "data"
RAW = DATA / "raw"
RAW_FDA = RAW / "dafdata20260313"
PROCESSED = DATA / "processed"
OUTPUT = ROOT / "output"

PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {ROOT}")
print(f"FDA extract: {RAW_FDA}")


Repo root: /Users/alexdelatorre/Desktop/econ580-thesis
FDA extract: /Users/alexdelatorre/Desktop/econ580-thesis/data/raw/dafdata20260313


## Source and extract check

We begin by confirming that the extracted FDA folder exists and that all expected text files are present. This is a basic but important reproducibility check: the notebook should fail early if the raw extract is incomplete.


In [2]:
expected_files = [
    "Applications.txt",
    "Submissions.txt",
    "Products.txt",
    "SubmissionClass_Lookup.txt",
    "SubmissionPropertyType.txt",
    "Join_Submission_ActionTypes_Lookup.txt",
    "ActionTypes_Lookup.txt",
    "ApplicationDocs.txt",
    "ApplicationsDocsType_Lookup.txt",
    "MarketingStatus.txt",
    "MarketingStatus_Lookup.txt",
    "TE.txt",
]

print("Extract folder exists:", RAW_FDA.exists())
print("Extract folder:", RAW_FDA)
print()

present_files = sorted(p.name for p in RAW_FDA.glob("*.txt"))
print("Files present in extract:")
for name in present_files:
    print("-", name)

missing_files = sorted(set(expected_files) - set(present_files))
print()
print("Missing expected files:", missing_files if missing_files else "None")


Extract folder exists: True
Extract folder: /Users/alexdelatorre/Desktop/econ580-thesis/data/raw/dafdata20260313

Files present in extract:
- ActionTypes_Lookup.txt
- ApplicationDocs.txt
- Applications.txt
- ApplicationsDocsType_Lookup.txt
- Join_Submission_ActionTypes_Lookup.txt
- MarketingStatus.txt
- MarketingStatus_Lookup.txt
- Products.txt
- SubmissionClass_Lookup.txt
- SubmissionPropertyType.txt
- Submissions.txt
- TE.txt

Missing expected files: None


## Helper functions

The extract is not perfectly clean. In particular:

- `Submissions.txt` requires a Latin-1 fallback
- `ApplicationDocs.txt` contains **one malformed line** with an extra empty tab-delimited field
- identifiers should remain as strings
- several child tables need aggregation before they can be joined safely

The helper functions below make those decisions explicit.


In [3]:
def count_data_rows(path: Path, encoding: str = "latin-1") -> int:
    with path.open("r", encoding=encoding, newline="") as f:
        return max(sum(1 for _ in f) - 1, 0)


def read_application_docs_table(path: Path) -> pd.DataFrame:
    """Read ApplicationDocs.txt with one documented row repair.

    The local 20260313 extract has exactly one row with 9 fields instead of 8 because
    there is an extra empty tab before the date field. We repair that one pattern by
    dropping the empty penultimate field and keeping the final date value.
    """
    rows = []
    with path.open("r", encoding="latin-1", newline="") as f:
        reader = csv.reader(f, delimiter="	")
        header = next(reader)
        for i, row in enumerate(reader, start=2):
            if len(row) == len(header):
                rows.append(row)
            elif len(row) == len(header) + 1 and row[-2] == "":
                rows.append(row[:-2] + [row[-1]])
            else:
                raise ValueError(
                    f"Unexpected field count in ApplicationDocs.txt at line {i}: {len(row)}"
                )
    return pd.DataFrame(rows, columns=header)


def read_fda_table(path: Path) -> pd.DataFrame:
    if path.name == "ApplicationDocs.txt":
        return read_application_docs_table(path)

    last_error = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, sep="	", dtype=str, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace("﻿", "", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def collapse_unique(values) -> pd._libs.missing.NAType | str:
    ordered = []
    seen = set()
    for value in pd.Series(list(values), dtype="string").dropna():
        value = value.strip()
        if value and value not in seen:
            ordered.append(value)
            seen.add(value)
    return "; ".join(ordered) if ordered else pd.NA


def standardize_review_priority(s: pd.Series) -> pd.Series:
    raw = clean_string_series(s).str.upper()
    out = pd.Series("OTHER", index=s.index, dtype="string")
    out.loc[raw.isna() | raw.isin(["UNKNOWN", "N/A"])] = "UNKNOWN"
    out.loc[raw.eq("STANDARD")] = "STANDARD"
    out.loc[raw.eq("PRIORITY")] = "PRIORITY"
    return out


def standardize_appl_type(s: pd.Series) -> pd.Series:
    raw = clean_string_series(s).str.upper()
    out = pd.Series("OTHER", index=s.index, dtype="string")
    out.loc[raw.isna()] = "UNKNOWN"
    out.loc[raw.isin(["NDA", "ANDA", "BLA"])] = raw[raw.isin(["NDA", "ANDA", "BLA"])]
    return out


def audit_table(name: str, df: pd.DataFrame, key_cols=None, sample_rows: int = 3):
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
    if key_cols is not None:
        dup_count = int(df.duplicated(key_cols).sum())
        print("Duplicate rows on key", key_cols, ":", dup_count)
    print()
    display(df.head(sample_rows))


## Load all 12 raw tables

We now load the entire extract into memory. This notebook intentionally works from the full set of text files rather than prematurely limiting attention to one or two "core" tables.


In [4]:
tables = {name: read_fda_table(RAW_FDA / name) for name in expected_files}

summary_rows = []
for name in expected_files:
    path = RAW_FDA / name
    df = tables[name]
    summary_rows.append({
        "file": name,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "on_disk_data_rows": count_data_rows(path),
    })

raw_table_summary = pd.DataFrame(summary_rows)
display(raw_table_summary)


,file,n_rows,n_cols,on_disk_data_rows
0,Applications.txt,28923,4,28923
1,Submissions.txt,191265,8,191265
2,Products.txt,50899,8,50899
3,SubmissionClass_Lookup.txt,28,3,28
4,SubmissionPropertyType.txt,308681,5,308681
5,Join_Submission_ActionTypes_Lookup.txt,206329,5,206329
6,ActionTypes_Lookup.txt,59,4,59
7,ApplicationDocs.txt,79456,8,79456
8,ApplicationsDocsType_Lookup.txt,62,2,62
9,MarketingStatus.txt,51517,3,51517


## Audit: `Applications.txt`

**Aim.** Audit the application-level file. This should tell us the application universe, basic application type, and sponsor information.


In [5]:
df = tables["Applications.txt"].copy()
audit_table("Applications.txt", df, key_cols=["ApplNo"])

df = tables["Applications.txt"].copy()
audit_table("Applications.txt", df, key_cols=["ApplNo"])
print("ApplType value counts")
print(df["ApplType"].astype("string").str.strip().str.upper().value_counts(dropna=False))
print()
print("Missing SponsorName:", int(df["SponsorName"].isna().sum()))
print("Missing ApplPublicNotes:", int(df["ApplPublicNotes"].isna().sum()))


Applications.txt: 28,923 rows x 4 columns
Columns: ['ApplNo', 'ApplType', 'ApplPublicNotes', 'SponsorName']
Duplicate rows on key ['ApplNo'] : 0



,ApplNo,ApplType,ApplPublicNotes,SponsorName
0,000004,NDA,NaN,PHARMICS
1,000159,NDA,NaN,LILLY
2,000552,NDA,NaN,ASPEN GLOBAL INC


Applications.txt: 28,923 rows x 4 columns
Columns: ['ApplNo', 'ApplType', 'ApplPublicNotes', 'SponsorName']
Duplicate rows on key ['ApplNo'] : 0



,ApplNo,ApplType,ApplPublicNotes,SponsorName
0,000004,NDA,NaN,PHARMICS
1,000159,NDA,NaN,LILLY
2,000552,NDA,NaN,ASPEN GLOBAL INC


ApplType value counts
ApplType
ANDA    22626
NDA      5832
BLA       465
Name: count, dtype: Int64

Missing SponsorName: 0
Missing ApplPublicNotes: 28922


**Takeaway.** Findings: `Applications.txt` is the application-level parent table with **28,923 unique `ApplNo` values**. In this extract, `ApplType` is overwhelmingly `ANDA`, followed by `NDA`, with a smaller `BLA` segment. It contains sponsor information but almost no public notes.


## Audit: `Submissions.txt`

**Aim.** Audit the submission-event file. This is the table that records original submissions and supplements, along with status dates and review-priority information.


In [6]:
df = tables["Submissions.txt"].copy()
audit_table("Submissions.txt", df, key_cols=["ApplNo", "SubmissionType", "SubmissionNo"])

df = tables["Submissions.txt"].copy()
audit_table("Submissions.txt", df, key_cols=["ApplNo", "SubmissionType", "SubmissionNo"])
print("SubmissionType value counts (trimmed)")
print(df["SubmissionType"].astype("string").str.strip().str.upper().value_counts(dropna=False))
print()
print("SubmissionStatus value counts (trimmed)")
print(df["SubmissionStatus"].astype("string").str.strip().str.upper().value_counts(dropna=False))
print()
dates = pd.to_datetime(df["SubmissionStatusDate"], errors="coerce")
print("SubmissionStatusDate min:", dates.min())
print("SubmissionStatusDate max:", dates.max())


Submissions.txt: 191,265 rows x 8 columns
Columns: ['ApplNo', 'SubmissionClassCodeID', 'SubmissionType', 'SubmissionNo', 'SubmissionStatus', 'SubmissionStatusDate', 'SubmissionsPublicNotes', 'ReviewPriority']
Duplicate rows on key ['ApplNo', 'SubmissionType', 'SubmissionNo'] : 0



,ApplNo,SubmissionClassCodeID,SubmissionType,SubmissionNo,SubmissionStatus,SubmissionStatusDate,SubmissionsPublicNotes,ReviewPriority
0,000004,19,ORIG,1,AP,1969-07-16 00:00:00,NaN,UNKNOWN
1,000004,3,SUPPL,10,AP,1980-05-08 00:00:00,NaN,NaN
2,000004,3,SUPPL,11,AP,1987-05-26 00:00:00,NaN,NaN


Submissions.txt: 191,265 rows x 8 columns
Columns: ['ApplNo', 'SubmissionClassCodeID', 'SubmissionType', 'SubmissionNo', 'SubmissionStatus', 'SubmissionStatusDate', 'SubmissionsPublicNotes', 'ReviewPriority']
Duplicate rows on key ['ApplNo', 'SubmissionType', 'SubmissionNo'] : 0



,ApplNo,SubmissionClassCodeID,SubmissionType,SubmissionNo,SubmissionStatus,SubmissionStatusDate,SubmissionsPublicNotes,ReviewPriority
0,000004,19,ORIG,1,AP,1969-07-16 00:00:00,NaN,UNKNOWN
1,000004,3,SUPPL,10,AP,1980-05-08 00:00:00,NaN,NaN
2,000004,3,SUPPL,11,AP,1987-05-26 00:00:00,NaN,NaN


SubmissionType value counts (trimmed)
SubmissionType
SUPPL    163831
ORIG      27434
Name: count, dtype: Int64

SubmissionStatus value counts (trimmed)
SubmissionStatus
AP      190051
TA        1213
<NA>         1
Name: count, dtype: Int64

SubmissionStatusDate min: 1900-01-01 00:00:00
SubmissionStatusDate max: 2026-03-11 00:00:00


**Takeaway.** Findings: `Submissions.txt` is the natural backbone table for a master panel because it has **191,265 unique submission events**. Most rows are `SUPPL`, not `ORIG`. Status values are overwhelmingly `AP`, with a smaller `TA` group and almost no other statuses, which is a strong sign that this Drugs@FDA extract is approval-centered rather than a full universe of rejected applications.


## Audit: `Products.txt`

**Aim.** Audit the product-level file. This is where drug identity, active ingredient, formulation, and strength live.


In [7]:
df = tables["Products.txt"].copy()
audit_table("Products.txt", df, key_cols=["ApplNo", "ProductNo"])

df = tables["Products.txt"].copy()
audit_table("Products.txt", df, key_cols=["ApplNo", "ProductNo"])
print("Unique applications represented in Products.txt:", int(df["ApplNo"].nunique()))
print()
print("Top dosage forms")
print(df["Form"].astype("string").str.strip().value_counts(dropna=False).head(10))


Products.txt: 50,899 rows x 8 columns
Columns: ['ApplNo', 'ProductNo', 'Form', 'Strength', 'ReferenceDrug', 'DrugName', 'ActiveIngredient', 'ReferenceStandard']
Duplicate rows on key ['ApplNo', 'ProductNo'] : 0



,ApplNo,ProductNo,Form,Strength,ReferenceDrug,DrugName,ActiveIngredient,ReferenceStandard
0,000004,004,SOLUTION/DROPS;OPHTHALMIC,1%,0,PAREDRINE,HYDROXYAMPHETAMINE HYDROBROMIDE,0
1,000159,001,TABLET;ORAL,500MG,0,SULFAPYRIDINE,SULFAPYRIDINE,0
2,000552,001,INJECTABLE;INJECTION,"20,000 UNITS/ML",0,LIQUAEMIN SODIUM,HEPARIN SODIUM,0


Products.txt: 50,899 rows x 8 columns
Columns: ['ApplNo', 'ProductNo', 'Form', 'Strength', 'ReferenceDrug', 'DrugName', 'ActiveIngredient', 'ReferenceStandard']
Duplicate rows on key ['ApplNo', 'ProductNo'] : 0



,ApplNo,ProductNo,Form,Strength,ReferenceDrug,DrugName,ActiveIngredient,ReferenceStandard
0,000004,004,SOLUTION/DROPS;OPHTHALMIC,1%,0,PAREDRINE,HYDROXYAMPHETAMINE HYDROBROMIDE,0
1,000159,001,TABLET;ORAL,500MG,0,SULFAPYRIDINE,SULFAPYRIDINE,0
2,000552,001,INJECTABLE;INJECTION,"20,000 UNITS/ML",0,LIQUAEMIN SODIUM,HEPARIN SODIUM,0


Unique applications represented in Products.txt: 28571

Top dosage forms
Form
TABLET;ORAL                       19951
INJECTABLE;INJECTION               7471
CAPSULE;ORAL                       5362
TABLET, EXTENDED RELEASE;ORAL      2617
CAPSULE, EXTENDED RELEASE;ORAL     1226
SOLUTION;INTRAVENOUS                996
SOLUTION;ORAL                       740
INJECTABLE;INTRAVENOUS              609
CREAM;TOPICAL                       601
SOLUTION/DROPS;OPHTHALMIC           594
Name: count, dtype: Int64


**Takeaway.** Findings: `Products.txt` is the main source of drug identity. It has **50,899 unique application-product rows** and is many-to-one with applications. That means it cannot be merged directly onto submission events without creating duplicates.


## Audit: `SubmissionClass_Lookup.txt`

**Aim.** Audit the lookup that decodes `SubmissionClassCodeID` from the submissions table.


In [8]:
df = tables["SubmissionClass_Lookup.txt"].copy()
audit_table("SubmissionClass_Lookup.txt", df, key_cols=["SubmissionClassCodeID"])

df = tables["SubmissionClass_Lookup.txt"].copy()
audit_table("SubmissionClass_Lookup.txt", df, key_cols=["SubmissionClassCodeID"])
display(df.sort_values("SubmissionClassCodeID"))


SubmissionClass_Lookup.txt: 28 rows x 3 columns
Columns: ['SubmissionClassCodeID', 'SubmissionClassCode', 'SubmissionClassCodeDescription']
Duplicate rows on key ['SubmissionClassCodeID'] : 0



,SubmissionClassCodeID,SubmissionClassCode,SubmissionClassCodeDescription
0,1,BIOEQUIV,Bioequivalence
1,2,EFFICACY,Efficacy
2,3,LABELING,Labeling


SubmissionClass_Lookup.txt: 28 rows x 3 columns
Columns: ['SubmissionClassCodeID', 'SubmissionClassCode', 'SubmissionClassCodeDescription']
Duplicate rows on key ['SubmissionClassCodeID'] : 0



,SubmissionClassCodeID,SubmissionClassCode,SubmissionClassCodeDescription
0,1,BIOEQUIV,Bioequivalence
1,2,EFFICACY,Efficacy
2,3,LABELING,Labeling


,SubmissionClassCodeID,SubmissionClassCode,SubmissionClassCodeDescription
0,1,BIOEQUIV,Bioequivalence
9,10,TYPE 2/3,Type 2 - New Active Ingredient and Type 3 - Ne...
10,11,TYPE 2/4,Type 2 New Active Ingredient and Type 4 New Co...
11,12,TYPE 3,Type 3 - New Dosage Form
12,13,TYPE 3/4,Type 3 - New Dosage Form and Type 4 - New Comb...
13,14,TYPE 4,Type 4 - New Combination
14,15,TYPE 5,Type 5 - New Formulation or New Manufacturer
15,16,TYPE 6,Type 6 - New Indication (no longer used)
16,17,TYPE 7,Type 7 - Drug Already Marketed without Approve...
17,18,TYPE 8,Type 8 - Partial Rx to OTC Switch


**Takeaway.** Findings: this lookup has **28 rows** and is necessary for interpreting the regulatory class attached to a submission event. It contains categories such as `TYPE 1`, `TYPE 3`, `TYPE 5`, `REMS`, and `BIOSIMILAR`.


## Audit: `SubmissionPropertyType.txt`

**Aim.** Audit the submission-property file. This table appears to attach one or more properties to a submission event.


In [9]:
df = tables["SubmissionPropertyType.txt"].copy()
audit_table("SubmissionPropertyType.txt", df, key_cols=["ApplNo", "SubmissionType", "SubmissionNo", "SubmissionPropertyTypeID"])

df = tables["SubmissionPropertyType.txt"].copy()
audit_table("SubmissionPropertyType.txt", df, key_cols=["ApplNo", "SubmissionType", "SubmissionNo", "SubmissionPropertyTypeID"])
print("SubmissionPropertyTypeCode value counts")
print(df["SubmissionPropertyTypeCode"].astype("string").str.strip().value_counts(dropna=False).head(10))
print()
print("Unique submission-event keys represented:", int(df[["ApplNo", "SubmissionType", "SubmissionNo"]].drop_duplicates().shape[0]))


SubmissionPropertyType.txt: 308,681 rows x 5 columns
Columns: ['ApplNo', 'SubmissionType', 'SubmissionNo', 'SubmissionPropertyTypeCode', 'SubmissionPropertyTypeID']
Duplicate rows on key ['ApplNo', 'SubmissionType', 'SubmissionNo', 'SubmissionPropertyTypeID'] : 0



,ApplNo,SubmissionType,SubmissionNo,SubmissionPropertyTypeCode,SubmissionPropertyTypeID
0,000159,ORIG,1,Null,0
1,000159,SUPPL,3,Null,0
2,000159,SUPPL,4,Null,0


SubmissionPropertyType.txt: 308,681 rows x 5 columns
Columns: ['ApplNo', 'SubmissionType', 'SubmissionNo', 'SubmissionPropertyTypeCode', 'SubmissionPropertyTypeID']
Duplicate rows on key ['ApplNo', 'SubmissionType', 'SubmissionNo', 'SubmissionPropertyTypeID'] : 0



,ApplNo,SubmissionType,SubmissionNo,SubmissionPropertyTypeCode,SubmissionPropertyTypeID
0,000159,ORIG,1,Null,0
1,000159,SUPPL,3,Null,0
2,000159,SUPPL,4,Null,0


SubmissionPropertyTypeCode value counts
SubmissionPropertyTypeCode
Null      305251
Orphan      3430
Name: count, dtype: Int64

Unique submission-event keys represented: 143079


**Takeaway.** Findings: `SubmissionPropertyType.txt` is many-to-one with submissions. The text code is mostly `Null`, while `Orphan` appears in a smaller set of rows. Because no local lookup for `SubmissionPropertyTypeID` is present in this extract, we should carry those IDs through but interpret them cautiously.


## Audit: `Join_Submission_ActionTypes_Lookup.txt`

**Aim.** Audit the join table that maps submissions to action-type codes.


In [10]:
df = tables["Join_Submission_ActionTypes_Lookup.txt"].copy()
audit_table("Join_Submission_ActionTypes_Lookup.txt", df, key_cols=["j_submissionActionTypeID"])

df = tables["Join_Submission_ActionTypes_Lookup.txt"].copy()
audit_table("Join_Submission_ActionTypes_Lookup.txt", df, key_cols=["j_submissionActionTypeID"])
print("Unique submission-event keys represented:", int(df[["ApplNo", "SubmissionType", "SubmissionNo"]].drop_duplicates().shape[0]))
print("Unique action IDs represented:", int(df["ActionTypes_LookupID"].nunique()))


Join_Submission_ActionTypes_Lookup.txt: 206,329 rows x 5 columns
Columns: ['SubmissionType', 'j_submissionActionTypeID', 'ApplNo', 'SubmissionNo', 'ActionTypes_LookupID']
Duplicate rows on key ['j_submissionActionTypeID'] : 0



,SubmissionType,j_submissionActionTypeID,ApplNo,SubmissionNo,ActionTypes_LookupID
0,SUPPL,1,017866,29,30
1,SUPPL,2,071450,5,30
2,SUPPL,3,017514,69,30


Join_Submission_ActionTypes_Lookup.txt: 206,329 rows x 5 columns
Columns: ['SubmissionType', 'j_submissionActionTypeID', 'ApplNo', 'SubmissionNo', 'ActionTypes_LookupID']
Duplicate rows on key ['j_submissionActionTypeID'] : 0



,SubmissionType,j_submissionActionTypeID,ApplNo,SubmissionNo,ActionTypes_LookupID
0,SUPPL,1,017866,29,30
1,SUPPL,2,071450,5,30
2,SUPPL,3,017514,69,30


Unique submission-event keys represented: 189122
Unique action IDs represented: 56


**Takeaway.** Findings: this is a many-to-many bridge between submission events and action types. It covers a very large share of submissions and gives more granular information about what kind of supplement or action a submission involved.


## Audit: `ActionTypes_Lookup.txt`

**Aim.** Audit the action-type lookup used to decode the join table above.


In [11]:
df = tables["ActionTypes_Lookup.txt"].copy()
audit_table("ActionTypes_Lookup.txt", df, key_cols=["ActionTypes_LookupID"])

df = tables["ActionTypes_Lookup.txt"].copy()
audit_table("ActionTypes_Lookup.txt", df, key_cols=["ActionTypes_LookupID"])
display(df.head(15))


ActionTypes_Lookup.txt: 59 rows x 4 columns
Columns: ['ActionTypes_LookupID', 'ActionTypes_LookupDescription', 'SupplCategoryLevel1Code', 'SupplCategoryLevel2Code']
Duplicate rows on key ['ActionTypes_LookupID'] : 0



,ActionTypes_LookupID,ActionTypes_LookupDescription,SupplCategoryLevel1Code,SupplCategoryLevel2Code
0,1,Bioequivalence,BIOEQUIV,NaN
1,2,Efficacy,EFFICACY,NOT APPLICABLE
2,3,Efficacy-Accelerated Approval,EFFICACY,ACCEL APP


ActionTypes_Lookup.txt: 59 rows x 4 columns
Columns: ['ActionTypes_LookupID', 'ActionTypes_LookupDescription', 'SupplCategoryLevel1Code', 'SupplCategoryLevel2Code']
Duplicate rows on key ['ActionTypes_LookupID'] : 0



,ActionTypes_LookupID,ActionTypes_LookupDescription,SupplCategoryLevel1Code,SupplCategoryLevel2Code
0,1,Bioequivalence,BIOEQUIV,NaN
1,2,Efficacy,EFFICACY,NOT APPLICABLE
2,3,Efficacy-Accelerated Approval,EFFICACY,ACCEL APP


,ActionTypes_LookupID,ActionTypes_LookupDescription,SupplCategoryLevel1Code,SupplCategoryLevel2Code
0,1,Bioequivalence,BIOEQUIV,NaN
1,2,Efficacy,EFFICACY,NOT APPLICABLE
2,3,Efficacy-Accelerated Approval,EFFICACY,ACCEL APP
3,4,Efficacy-Accelerated Approval Confirmatory Study,EFFICACY,COMP EFF
4,5,Efficacy-Labeling Change With Clinical Data,EFFICACY,LABEL W CLIN
5,6,Efficacy-Manufacturing Change With Clinical Data,EFFICACY,MANU W CLIN
6,7,Efficacy-New Dosing Regimen,EFFICACY,DOSING
7,8,Efficacy-New Indication,EFFICACY,INDICATION
8,9,Efficacy-New Patient Population,EFFICACY,PAT POPUL
9,10,Efficacy-New Route Of Administration,EFFICACY,ROUTE


**Takeaway.** Findings: the lookup contains **59 action codes**. Some descriptions are duplicated, and a few are blank or uninformative. We can still use the descriptions and supplementary category codes, but they should be treated as administrative labels rather than perfect analytic categories.


## Audit: `ApplicationDocs.txt`

**Aim.** Audit the document-metadata file. This table records linked FDA documents rather than the document contents themselves.


In [12]:
df = tables["ApplicationDocs.txt"].copy()
audit_table("ApplicationDocs.txt", df, key_cols=["ApplicationDocsID"])

df = tables["ApplicationDocs.txt"].copy()
audit_table("ApplicationDocs.txt", df, key_cols=["ApplicationDocsID"])
print("ApplicationDocsTypeID top counts")
print(df["ApplicationDocsTypeID"].value_counts(dropna=False).head(10))
print()
dates = pd.to_datetime(df["ApplicationDocsDate"], errors="coerce")
print("ApplicationDocsDate min:", dates.min())
print("ApplicationDocsDate max:", dates.max())


ApplicationDocs.txt: 79,456 rows x 8 columns
Columns: ['ApplicationDocsID', 'ApplicationDocsTypeID', 'ApplNo', 'SubmissionType', 'SubmissionNo', 'ApplicationDocsTitle', 'ApplicationDocsURL', 'ApplicationDocsDate']
Duplicate rows on key ['ApplicationDocsID'] : 0



,ApplicationDocsID,ApplicationDocsTypeID,ApplNo,SubmissionType,SubmissionNo,ApplicationDocsTitle,ApplicationDocsURL,ApplicationDocsDate
0,1,1,004782,SUPPL,125,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2003-07-28 00:00:00
1,2,1,004782,SUPPL,128,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2002-11-27 00:00:00
2,3,1,004782,SUPPL,130,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2003-05-16 00:00:00


ApplicationDocs.txt: 79,456 rows x 8 columns
Columns: ['ApplicationDocsID', 'ApplicationDocsTypeID', 'ApplNo', 'SubmissionType', 'SubmissionNo', 'ApplicationDocsTitle', 'ApplicationDocsURL', 'ApplicationDocsDate']
Duplicate rows on key ['ApplicationDocsID'] : 0



,ApplicationDocsID,ApplicationDocsTypeID,ApplNo,SubmissionType,SubmissionNo,ApplicationDocsTitle,ApplicationDocsURL,ApplicationDocsDate
0,1,1,004782,SUPPL,125,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2003-07-28 00:00:00
1,2,1,004782,SUPPL,128,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2002-11-27 00:00:00
2,3,1,004782,SUPPL,130,0,http://www.accessdata.fda.gov/drugsatfda_docs/...,2003-05-16 00:00:00


ApplicationDocsTypeID top counts
ApplicationDocsTypeID
1     36487
2     29182
3      7766
8      1192
10      861
21      767
20      559
19      497
18      467
14      456
Name: count, dtype: int64

ApplicationDocsDate min: 1900-01-01 00:00:00
ApplicationDocsDate max: 2026-03-12 00:00:00


**Takeaway.** Findings: `ApplicationDocs.txt` contains **79,456 document rows** and required a custom reader because the local extract has **one malformed row** with an extra empty field. This file is valuable, but it is document metadata, not full text. It can be summarized at the submission level and at the application level.


## Audit: `ApplicationsDocsType_Lookup.txt`

**Aim.** Audit the document-type lookup used to interpret `ApplicationDocsTypeID`.


In [13]:
df = tables["ApplicationsDocsType_Lookup.txt"].copy()
audit_table("ApplicationsDocsType_Lookup.txt", df, key_cols=["ApplicationDocsType_Lookup_ID"])

df = tables["ApplicationsDocsType_Lookup.txt"].copy()
audit_table("ApplicationsDocsType_Lookup.txt", df, key_cols=["ApplicationDocsType_Lookup_ID"])
display(df.head(20))


ApplicationsDocsType_Lookup.txt: 62 rows x 2 columns
Columns: ['ApplicationDocsType_Lookup_ID', 'ApplicationDocsType_Lookup_Description']
Duplicate rows on key ['ApplicationDocsType_Lookup_ID'] : 0



,ApplicationDocsType_Lookup_ID,ApplicationDocsType_Lookup_Description
0,1,Letter
1,2,Label
2,3,Review


ApplicationsDocsType_Lookup.txt: 62 rows x 2 columns
Columns: ['ApplicationDocsType_Lookup_ID', 'ApplicationDocsType_Lookup_Description']
Duplicate rows on key ['ApplicationDocsType_Lookup_ID'] : 0



,ApplicationDocsType_Lookup_ID,ApplicationDocsType_Lookup_Description
0,1,Letter
1,2,Label
2,3,Review


,ApplicationDocsType_Lookup_ID,ApplicationDocsType_Lookup_Description
0,1,Letter
1,2,Label
2,3,Review
3,4,FDA Talk Paper
4,5,FDA Press Release
5,6,Patient Package Insert
6,7,Dear Health Professional Letter
7,8,Medication Guide
8,9,Withdrawal Notice
9,10,Other Important Information from FDA


**Takeaway.** Findings: this lookup contains **62 document-type labels**, including letters, labels, reviews, REMS documents, and several pediatric review variants. It lets us summarize document coverage without needing to carry every raw URL into the master file.


## Audit: `MarketingStatus.txt`

**Aim.** Audit the product-level marketing-status table.


In [14]:
df = tables["MarketingStatus.txt"].copy()
audit_table("MarketingStatus.txt", df, key_cols=["ApplNo", "ProductNo"])

df = tables["MarketingStatus.txt"].copy()
audit_table("MarketingStatus.txt", df, key_cols=["ApplNo", "ProductNo"])
print("MarketingStatusID value counts")
print(df["MarketingStatusID"].value_counts(dropna=False))


MarketingStatus.txt: 51,517 rows x 3 columns
Columns: ['MarketingStatusID', 'ApplNo', 'ProductNo']
Duplicate rows on key ['ApplNo', 'ProductNo'] : 0



,MarketingStatusID,ApplNo,ProductNo
0,3,000004,004
1,3,000159,001
2,3,000552,001


MarketingStatus.txt: 51,517 rows x 3 columns
Columns: ['MarketingStatusID', 'ApplNo', 'ProductNo']
Duplicate rows on key ['ApplNo', 'ProductNo'] : 0



,MarketingStatusID,ApplNo,ProductNo
0,3,000004,004
1,3,000159,001
2,3,000552,001


MarketingStatusID value counts
MarketingStatusID
1    25664
3    22989
4     2055
2      808
5        1
Name: count, dtype: int64


**Takeaway.** Findings: `MarketingStatus.txt` is product-level and maps application-product pairs to marketing-status IDs. It can enrich the master file, but it must first be aggregated to the application level if we want to avoid duplicating submission rows.


## Audit: `MarketingStatus_Lookup.txt`

**Aim.** Audit the marketing-status lookup.


In [15]:
df = tables["MarketingStatus_Lookup.txt"].copy()
audit_table("MarketingStatus_Lookup.txt", df, key_cols=["MarketingStatusID"])

df = tables["MarketingStatus_Lookup.txt"].copy()
audit_table("MarketingStatus_Lookup.txt", df, key_cols=["MarketingStatusID"])
display(df)


MarketingStatus_Lookup.txt: 5 rows x 2 columns
Columns: ['MarketingStatusID', 'MarketingStatusDescription']
Duplicate rows on key ['MarketingStatusID'] : 0



,MarketingStatusID,MarketingStatusDescription
0,1,Prescription
1,2,Over-the-counter
2,3,Discontinued


MarketingStatus_Lookup.txt: 5 rows x 2 columns
Columns: ['MarketingStatusID', 'MarketingStatusDescription']
Duplicate rows on key ['MarketingStatusID'] : 0



,MarketingStatusID,MarketingStatusDescription
0,1,Prescription
1,2,Over-the-counter
2,3,Discontinued


,MarketingStatusID,MarketingStatusDescription
0,1,Prescription
1,2,Over-the-counter
2,3,Discontinued
3,4,None (Tentative Approval)
4,5,For Further Manufacturing Use


**Takeaway.** Findings: this lookup is small but important. It tells us that the raw status codes correspond to categories such as `Prescription`, `Over-the-counter`, `Discontinued`, `None (Tentative Approval)`, and `For Further Manufacturing Use`.


## Audit: `TE.txt`

**Aim.** Audit the therapeutic-equivalence file.


In [16]:
df = tables["TE.txt"].copy()
audit_table("TE.txt", df, key_cols=["ApplNo", "ProductNo", "MarketingStatusID", "TECode"])

df = tables["TE.txt"].copy()
audit_table("TE.txt", df, key_cols=["ApplNo", "ProductNo", "MarketingStatusID", "TECode"])
print("Top TE codes")
print(df["TECode"].value_counts(dropna=False).head(15))
print()
print("Unique application-product pairs represented:", int(df[["ApplNo", "ProductNo"]].drop_duplicates().shape[0]))


TE.txt: 25,804 rows x 4 columns
Columns: ['ApplNo', 'ProductNo', 'MarketingStatusID', 'TECode']
Duplicate rows on key ['ApplNo', 'ProductNo', 'MarketingStatusID', 'TECode'] : 0



,ApplNo,ProductNo,MarketingStatusID,TECode
0,003444,001,1,AA
1,004782,001,1,AB
2,004782,003,1,AB


TE.txt: 25,804 rows x 4 columns
Columns: ['ApplNo', 'ProductNo', 'MarketingStatusID', 'TECode']
Duplicate rows on key ['ApplNo', 'ProductNo', 'MarketingStatusID', 'TECode'] : 0



,ApplNo,ProductNo,MarketingStatusID,TECode
0,003444,001,1,AA
1,004782,001,1,AB
2,004782,003,1,AB


Top TE codes
TECode
AB     17300
AP      4368
AA      1249
AB1      755
AT       665
AB2      450
AB3      273
AN       154
BX       119
AB4      112
AO        88
AP1       70
AT1       54
TBD       44
AP2       38
Name: count, dtype: int64

Unique application-product pairs represented: 25490


**Takeaway.** Findings: `TE.txt` is a **partial** product-level table, not a full inventory of products. It is useful as enrichment for some products, but it should never define the universe of applications or submission events.


## Synthesis after the table-by-table audit

After inspecting the raw files one by one, the most defensible design is:

- use **one row per submission event** as the master unit of observation
- start from `Submissions.txt`
- keep **all** submission rows, not just `ORIG` and not just `AP`
- treat `Applications.txt` as the application-level parent
- treat `Products.txt`, `MarketingStatus.txt`, and `TE.txt` as child tables that must be aggregated before merge
- treat `SubmissionPropertyType.txt`, `Join_Submission_ActionTypes_Lookup.txt`, and `ApplicationDocs.txt` as child tables that must be aggregated at the submission-event level

A second important conclusion is substantive: this Drugs@FDA extract does **not** appear to be a full universe of failed or rejected submissions. In the local snapshot, submission statuses are overwhelmingly `AP` with a smaller `TA` group. That means the raw data are very informative about approvals, tentative approvals, supplements, products, and regulatory metadata, but much less informative about outright non-approval outcomes.


## Build the master dataset

The next steps create a single master panel with **one row per submission event** and then merge in child-table information using conservative aggregates. This keeps all the available submission rows while avoiding duplicate inflation from one-to-many joins.


In [17]:
# Copy the raw tables so the build steps below do not mutate the originals used in the audits.
applications = tables["Applications.txt"].copy()
submissions = tables["Submissions.txt"].copy()
products = tables["Products.txt"].copy()
submission_class_lookup = tables["SubmissionClass_Lookup.txt"].copy()
submission_property_type = tables["SubmissionPropertyType.txt"].copy()
join_action = tables["Join_Submission_ActionTypes_Lookup.txt"].copy()
action_lookup = tables["ActionTypes_Lookup.txt"].copy()
application_docs = tables["ApplicationDocs.txt"].copy()
application_docs_type_lookup = tables["ApplicationsDocsType_Lookup.txt"].copy()
marketing_status = tables["MarketingStatus.txt"].copy()
marketing_status_lookup = tables["MarketingStatus_Lookup.txt"].copy()
te = tables["TE.txt"].copy()

# Standardize strings before merge logic.
for df in [
    applications,
    submissions,
    products,
    submission_class_lookup,
    submission_property_type,
    join_action,
    action_lookup,
    application_docs,
    application_docs_type_lookup,
    marketing_status,
    marketing_status_lookup,
    te,
]:
    for col in df.columns:
        df[col] = clean_string_series(df[col])

# Normalize the main coded fields used in filters and joins.
submissions["SubmissionType"] = submissions["SubmissionType"].str.upper()
submissions["SubmissionStatus"] = submissions["SubmissionStatus"].str.upper()
applications["ApplType"] = applications["ApplType"].str.upper()
submission_property_type["SubmissionType"] = submission_property_type["SubmissionType"].str.upper()
join_action["SubmissionType"] = join_action["SubmissionType"].str.upper()
application_docs["SubmissionType"] = application_docs["SubmissionType"].str.upper()

# Parse dates and treat the FDA placeholder date as missing.
submissions["SubmissionStatusDate"] = pd.to_datetime(submissions["SubmissionStatusDate"], errors="coerce")
submissions.loc[
    submissions["SubmissionStatusDate"] == pd.Timestamp("1900-01-01"),
    "SubmissionStatusDate",
] = pd.NaT

application_docs["ApplicationDocsDate"] = pd.to_datetime(application_docs["ApplicationDocsDate"], errors="coerce")
application_docs.loc[
    application_docs["ApplicationDocsDate"] == pd.Timestamp("1900-01-01"),
    "ApplicationDocsDate",
] = pd.NaT


### Build application-level aggregates

These aggregates are attached to every submission row for the same application. They bring in drug identity and application-level context without multiplying submission events.


In [18]:
products_agg = (
    products.sort_values(["ApplNo", "ProductNo"])
    .groupby("ApplNo", dropna=False)
    .agg(
        n_products=("ProductNo", "nunique"),
        ProductNo_list=("ProductNo", collapse_unique),
        DrugName_list=("DrugName", collapse_unique),
        ActiveIngredient_list=("ActiveIngredient", collapse_unique),
        Form_list=("Form", collapse_unique),
        Strength_list=("Strength", collapse_unique),
        ReferenceDrug_values=("ReferenceDrug", collapse_unique),
        ReferenceStandard_values=("ReferenceStandard", collapse_unique),
    )
    .reset_index()
)

marketing_status_agg = (
    marketing_status
    .merge(marketing_status_lookup, on="MarketingStatusID", how="left", validate="m:1")
    .groupby("ApplNo", dropna=False)
    .agg(
        n_marketing_status_rows=("ProductNo", "size"),
        MarketingStatusID_list=("MarketingStatusID", collapse_unique),
        MarketingStatusDescription_list=("MarketingStatusDescription", collapse_unique),
    )
    .reset_index()
)

te_agg = (
    te.groupby("ApplNo", dropna=False)
    .agg(
        n_te_rows=("TECode", "size"),
        TECode_list=("TECode", collapse_unique),
        TE_ProductNo_list=("ProductNo", collapse_unique),
        TE_MarketingStatusID_list=("MarketingStatusID", collapse_unique),
    )
    .reset_index()
)

print("Application-level aggregate coverage")
print({
    "applications_with_products": int(products_agg["ApplNo"].nunique()),
    "applications_with_marketing_status": int(marketing_status_agg["ApplNo"].nunique()),
    "applications_with_te_rows": int(te_agg["ApplNo"].nunique()),
})


Application-level aggregate coverage
{'applications_with_products': 28571, 'applications_with_marketing_status': 28906, 'applications_with_te_rows': 13102}


### Build submission-level child-table aggregates

These tables map directly to the submission-event key or are naturally summarized there. They let us keep submission properties, action types, and document metadata without duplicating rows.


In [19]:
submission_property_agg = (
    submission_property_type
    .groupby(["ApplNo", "SubmissionType", "SubmissionNo"], dropna=False)
    .agg(
        n_submission_property_rows=("SubmissionPropertyTypeID", "size"),
        SubmissionPropertyTypeCode_list=("SubmissionPropertyTypeCode", collapse_unique),
        SubmissionPropertyTypeID_list=("SubmissionPropertyTypeID", collapse_unique),
    )
    .reset_index()
)
submission_property_agg["has_orphan_property"] = (
    submission_property_agg["SubmissionPropertyTypeCode_list"]
    .fillna("")
    .str.contains("Orphan", regex=False)
)

action_type_agg = (
    join_action
    .merge(action_lookup, on="ActionTypes_LookupID", how="left", validate="m:1")
    .groupby(["ApplNo", "SubmissionType", "SubmissionNo"], dropna=False)
    .agg(
        n_action_rows=("ActionTypes_LookupID", "size"),
        ActionTypes_LookupID_list=("ActionTypes_LookupID", collapse_unique),
        ActionTypes_LookupDescription_list=("ActionTypes_LookupDescription", collapse_unique),
        SupplCategoryLevel1Code_list=("SupplCategoryLevel1Code", collapse_unique),
        SupplCategoryLevel2Code_list=("SupplCategoryLevel2Code", collapse_unique),
    )
    .reset_index()
)

doc_type_base = application_docs.merge(
    application_docs_type_lookup,
    left_on="ApplicationDocsTypeID",
    right_on="ApplicationDocsType_Lookup_ID",
    how="left",
    validate="m:1",
)

submission_docs_agg = (
    doc_type_base
    .groupby(["ApplNo", "SubmissionType", "SubmissionNo"], dropna=False)
    .agg(
        n_submission_docs=("ApplicationDocsID", "size"),
        ApplicationDocsTypeID_list=("ApplicationDocsTypeID", collapse_unique),
        ApplicationDocsTypeDescription_list=("ApplicationDocsType_Lookup_Description", collapse_unique),
        submission_doc_date_min=("ApplicationDocsDate", "min"),
        submission_doc_date_max=("ApplicationDocsDate", "max"),
    )
    .reset_index()
)

application_docs_agg = (
    doc_type_base
    .groupby("ApplNo", dropna=False)
    .agg(
        n_application_docs=("ApplicationDocsID", "size"),
        ApplicationDocsTypeDescription_app_level=("ApplicationDocsType_Lookup_Description", collapse_unique),
        application_doc_date_min=("ApplicationDocsDate", "min"),
        application_doc_date_max=("ApplicationDocsDate", "max"),
    )
    .reset_index()
)

print("Submission-level aggregate coverage")
print({
    "submission_keys_with_property_rows": int(submission_property_agg[["ApplNo", "SubmissionType", "SubmissionNo"]].drop_duplicates().shape[0]),
    "submission_keys_with_action_rows": int(action_type_agg[["ApplNo", "SubmissionType", "SubmissionNo"]].drop_duplicates().shape[0]),
    "submission_keys_with_doc_rows": int(submission_docs_agg[["ApplNo", "SubmissionType", "SubmissionNo"]].drop_duplicates().shape[0]),
})


Submission-level aggregate coverage
{'submission_keys_with_property_rows': 143079, 'submission_keys_with_action_rows': 189122, 'submission_keys_with_doc_rows': 40835}


### Build application-level timeline summaries from the submissions table

These are application-level summaries derived from `Submissions.txt` itself. They help us see the broader regulatory history of each application while still keeping the master dataset at the submission-event level.


In [20]:
submission_flags = submissions.assign(
    is_orig=submissions["SubmissionType"].eq("ORIG").fillna(False).astype(int),
    is_suppl=submissions["SubmissionType"].eq("SUPPL").fillna(False).astype(int),
    is_ap=submissions["SubmissionStatus"].eq("AP").fillna(False).astype(int),
    is_ta=submissions["SubmissionStatus"].eq("TA").fillna(False).astype(int),
)

application_timeline = (
    submissions
    .groupby("ApplNo", dropna=False)
    .agg(
        n_submissions_total=("SubmissionNo", "size"),
        first_submission_status_date=("SubmissionStatusDate", "min"),
        last_submission_status_date=("SubmissionStatusDate", "max"),
    )
    .reset_index()
    .merge(
        submission_flags.groupby("ApplNo", dropna=False)[["is_orig", "is_suppl", "is_ap", "is_ta"]]
        .sum()
        .reset_index()
        .rename(columns={
            "is_orig": "n_orig_submissions",
            "is_suppl": "n_supplement_submissions",
            "is_ap": "n_approved_submissions",
            "is_ta": "n_tentative_submissions",
        }),
        on="ApplNo",
        how="left",
        validate="m:1",
    )
)

first_approved_status_date = (
    submissions.loc[submissions["SubmissionStatus"].eq("AP")]
    .groupby("ApplNo", dropna=False)["SubmissionStatusDate"]
    .min()
    .reset_index(name="first_approved_status_date")
)

application_timeline = application_timeline.merge(
    first_approved_status_date,
    on="ApplNo",
    how="left",
    validate="m:1",
)

display(application_timeline.head())


,ApplNo,n_submissions_total,first_submission_status_date,last_submission_status_date,n_orig_submissions,n_supplement_submissions,n_approved_submissions,n_tentative_submissions,first_approved_status_date
0,000004,3,1969-07-16,1987-05-26,1,2,3,0,1969-07-16
1,000159,3,1939-03-09,1986-12-09,1,2,3,0,1939-03-09
2,000415,1,1939-02-27,1939-02-27,1,0,1,0,1939-02-27
3,000552,17,1939-02-09,2024-10-24,1,16,17,0,1939-02-09
4,000654,1,1939-05-04,1939-05-04,1,0,1,0,1939-05-04


### Merge the master submission-event panel

This is the core build step. The row count should remain exactly equal to the row count of `Submissions.txt`, because every merge is either many-to-one or one-to-one relative to the submission-event key.


In [21]:
fda_master = (
    submissions
    .merge(applications, on="ApplNo", how="left", validate="m:1")
    .merge(submission_class_lookup, on="SubmissionClassCodeID", how="left", validate="m:1")
    .merge(application_timeline, on="ApplNo", how="left", validate="m:1")
    .merge(products_agg, on="ApplNo", how="left", validate="m:1")
    .merge(marketing_status_agg, on="ApplNo", how="left", validate="m:1")
    .merge(te_agg, on="ApplNo", how="left", validate="m:1")
    .merge(submission_property_agg, on=["ApplNo", "SubmissionType", "SubmissionNo"], how="left", validate="m:1")
    .merge(action_type_agg, on=["ApplNo", "SubmissionType", "SubmissionNo"], how="left", validate="m:1")
    .merge(submission_docs_agg, on=["ApplNo", "SubmissionType", "SubmissionNo"], how="left", validate="m:1")
    .merge(application_docs_agg, on="ApplNo", how="left", validate="m:1")
    .copy()
)

# Derived variables that are convenient for downstream descriptive work.
fda_master["submission_status_year"] = fda_master["SubmissionStatusDate"].dt.year.astype("Int64")
fda_master["ReviewPriority_raw"] = fda_master["ReviewPriority"]
fda_master["ReviewPriority_clean"] = standardize_review_priority(fda_master["ReviewPriority"])
fda_master["ApplType_raw"] = fda_master["ApplType"]
fda_master["ApplType_clean"] = standardize_appl_type(fda_master["ApplType"])
fda_master["is_approved"] = fda_master["SubmissionStatus"].eq("AP")
fda_master["is_tentative_approval"] = fda_master["SubmissionStatus"].eq("TA")
fda_master["is_original_submission"] = fda_master["SubmissionType"].eq("ORIG")

# Keep the submission-event key unique and visible.
dup_submission_keys = int(
    fda_master.duplicated(["ApplNo", "SubmissionType", "SubmissionNo"]).sum()
)
print("Master rows:", len(fda_master))
print("Raw submission rows:", len(submissions))
print("Duplicate submission-event keys after merge:", dup_submission_keys)


Master rows: 191265
Raw submission rows: 191265
Duplicate submission-event keys after merge: 0


### Diagnostics on the full master dataset

At this point the key question is not "did we get a pretty subset?" The key question is whether the full master file is faithful to the raw extract and analytically usable. The checks below focus on row counts, status coverage, time coverage, and missingness in the main fields we are likely to use tomorrow.


In [22]:
# Format the date columns as strings for display and eventual CSV export.
for col in [
    "SubmissionStatusDate",
    "first_submission_status_date",
    "last_submission_status_date",
    "first_approved_status_date",
    "submission_doc_date_min",
    "submission_doc_date_max",
    "application_doc_date_min",
    "application_doc_date_max",
]:
    if col in fda_master.columns:
        fda_master[col] = pd.to_datetime(fda_master[col], errors="coerce").dt.strftime("%Y-%m-%d").astype("string")

print("Master dataset summary")
print({
    "rows": int(len(fda_master)),
    "columns": int(fda_master.shape[1]),
    "duplicate_submission_event_keys": dup_submission_keys,
    "approved_rows": int(fda_master["is_approved"].sum()),
    "tentative_rows": int(fda_master["is_tentative_approval"].sum()),
    "original_submission_rows": int(fda_master["is_original_submission"].sum()),
})

print()
print("SubmissionStatus counts")
print(fda_master["SubmissionStatus"].value_counts(dropna=False))

print()
print("SubmissionType counts")
print(fda_master["SubmissionType"].value_counts(dropna=False))

print()
print("Clean ApplType counts")
print(fda_master["ApplType_clean"].value_counts(dropna=False))

print()
print("Clean ReviewPriority counts")
print(fda_master["ReviewPriority_clean"].value_counts(dropna=False))

years = fda_master["submission_status_year"].dropna()
print()
print("Submission-status year range")
print({
    "min_year": int(years.min()) if len(years) else None,
    "max_year": int(years.max()) if len(years) else None,
})

coverage = {
    "rows_missing_application_match": int(fda_master["ApplType_raw"].isna().sum()),
    "rows_missing_submission_class_lookup": int(fda_master["SubmissionClassCode"].isna().sum()),
    "rows_missing_product_aggregate": int(fda_master["n_products"].isna().sum()),
    "rows_missing_marketing_status_aggregate": int(fda_master["MarketingStatusDescription_list"].isna().sum()),
    "rows_missing_te_aggregate": int(fda_master["TECode_list"].isna().sum()),
}
print()
print("Merge coverage checks")
for key, value in coverage.items():
    print(f"- {key}: {value}")

key_cols = [
    "ApplNo",
    "SubmissionType",
    "SubmissionNo",
    "SubmissionStatus",
    "SubmissionStatusDate",
    "submission_status_year",
    "ApplType_clean",
    "SponsorName",
    "DrugName_list",
    "ActiveIngredient_list",
    "SubmissionClassCode",
]

print()
print("Missingness in selected key fields")
display(
    pd.DataFrame({
        "missing_n": fda_master[key_cols].isna().sum(),
        "missing_share": fda_master[key_cols].isna().mean().round(4),
    }).sort_values(["missing_share", "missing_n"], ascending=False)
)

display(fda_master.head())


Master dataset summary
{'rows': 191265, 'columns': 62, 'duplicate_submission_event_keys': 0, 'approved_rows': 190051, 'tentative_rows': 1213, 'original_submission_rows': 27434}

SubmissionStatus counts
SubmissionStatus
AP      190051
TA        1213
<NA>         1
Name: count, dtype: Int64

SubmissionType counts
SubmissionType
SUPPL    163831
ORIG      27434
Name: count, dtype: Int64

Clean ApplType counts
ApplType_clean
ANDA       99441
NDA        80915
BLA         5548
UNKNOWN     5361
Name: count, dtype: Int64

Clean ReviewPriority counts
ReviewPriority_clean
STANDARD    93661
UNKNOWN     85550
PRIORITY    11092
OTHER         962
Name: count, dtype: Int64

Submission-status year range
{'min_year': 1939, 'max_year': 2026}

Merge coverage checks
- rows_missing_application_match: 5361
- rows_missing_submission_class_lookup: 12431
- rows_missing_product_aggregate: 5737
- rows_missing_marketing_status_aggregate: 5402
- rows_missing_te_aggregate: 104517

Missingness in selected key fields


,missing_n,missing_share
SubmissionClassCode,12431,0.065
DrugName_list,5737,0.030
ActiveIngredient_list,5737,0.030
SponsorName,5361,0.028
SubmissionStatusDate,6,0.000
submission_status_year,6,0.000
SubmissionStatus,1,0.000
ApplNo,0,0.000
SubmissionType,0,0.000
SubmissionNo,0,0.000


,ApplNo,SubmissionClassCodeID,SubmissionType,SubmissionNo,SubmissionStatus,SubmissionStatusDate,SubmissionsPublicNotes,ReviewPriority,ApplType,ApplPublicNotes,...,application_doc_date_min,application_doc_date_max,submission_status_year,ReviewPriority_raw,ReviewPriority_clean,ApplType_raw,ApplType_clean,is_approved,is_tentative_approval,is_original_submission
0,000004,19,ORIG,1,AP,1969-07-16,<NA>,UNKNOWN,NDA,<NA>,...,<NA>,<NA>,1969,UNKNOWN,UNKNOWN,NDA,NDA,True,False,True
1,000004,3,SUPPL,10,AP,1980-05-08,<NA>,<NA>,NDA,<NA>,...,<NA>,<NA>,1980,<NA>,UNKNOWN,NDA,NDA,True,False,False
2,000004,3,SUPPL,11,AP,1987-05-26,<NA>,<NA>,NDA,<NA>,...,<NA>,<NA>,1987,<NA>,UNKNOWN,NDA,NDA,True,False,False
3,000159,<NA>,ORIG,1,AP,1939-03-09,<NA>,<NA>,NDA,<NA>,...,<NA>,<NA>,1939,<NA>,UNKNOWN,NDA,NDA,True,False,True
4,000159,3,SUPPL,3,AP,1986-12-09,<NA>,<NA>,NDA,<NA>,...,<NA>,<NA>,1986,<NA>,UNKNOWN,NDA,NDA,True,False,False


## Export the full master panel

The export below is intentionally **not filtered** to `ORIG` and **not filtered** to `AP`. It saves the full submission-event panel because that is the cleanest way to preserve the available information for tomorrow's exploratory work.

Downstream subsets such as original approvals, tentative approvals, NDA-only rows, or approval-only rows should be derived from this master file, not hard-coded into the exported file itself.


In [23]:
export_cols = [
    "ApplNo",
    "ApplType_raw",
    "ApplType_clean",
    "ApplPublicNotes",
    "SponsorName",
    "SubmissionType",
    "SubmissionNo",
    "SubmissionStatus",
    "SubmissionStatusDate",
    "submission_status_year",
    "ReviewPriority_raw",
    "ReviewPriority_clean",
    "SubmissionClassCodeID",
    "SubmissionClassCode",
    "SubmissionClassCodeDescription",
    "is_approved",
    "is_tentative_approval",
    "is_original_submission",
    "n_submissions_total",
    "n_orig_submissions",
    "n_supplement_submissions",
    "n_approved_submissions",
    "n_tentative_submissions",
    "first_submission_status_date",
    "last_submission_status_date",
    "first_approved_status_date",
    "n_products",
    "ProductNo_list",
    "DrugName_list",
    "ActiveIngredient_list",
    "Form_list",
    "Strength_list",
    "ReferenceDrug_values",
    "ReferenceStandard_values",
    "n_marketing_status_rows",
    "MarketingStatusID_list",
    "MarketingStatusDescription_list",
    "n_te_rows",
    "TECode_list",
    "TE_ProductNo_list",
    "TE_MarketingStatusID_list",
    "n_submission_property_rows",
    "SubmissionPropertyTypeCode_list",
    "SubmissionPropertyTypeID_list",
    "has_orphan_property",
    "n_action_rows",
    "ActionTypes_LookupID_list",
    "ActionTypes_LookupDescription_list",
    "SupplCategoryLevel1Code_list",
    "SupplCategoryLevel2Code_list",
    "n_submission_docs",
    "ApplicationDocsTypeID_list",
    "ApplicationDocsTypeDescription_list",
    "submission_doc_date_min",
    "submission_doc_date_max",
    "n_application_docs",
    "ApplicationDocsTypeDescription_app_level",
    "application_doc_date_min",
    "application_doc_date_max",
]

export_df = fda_master[export_cols].copy()

# Keep the core identifiers as strings for safe downstream merges.
for col in ["ApplNo", "SubmissionType", "SubmissionNo"]:
    export_df[col] = export_df[col].astype("string")

export_path = PROCESSED / "fda_backbone.csv"
export_df.to_csv(export_path, index=False)

print("Saved full master submission-event panel to:")
print(export_path)
print()
print({
    "rows": int(len(export_df)),
    "columns": int(export_df.shape[1]),
    "duplicate_submission_event_keys": int(export_df.duplicated(["ApplNo", "SubmissionType", "SubmissionNo"]).sum()),
})


Saved full master submission-event panel to:
/Users/alexdelatorre/Desktop/econ580-thesis/data/processed/fda_backbone.csv

{'rows': 191265, 'columns': 59, 'duplicate_submission_event_keys': 0}
